In [3]:
import os
import re
import json
import time
import pandas as pd
from openai import OpenAI

In [4]:
import pandas as pd

file_path = "data/Reddit_askdocs_2k.csv"  # replace with exact CSV filename

df = pd.read_csv(file_path)

print(df.shape)
df.head()

(2000, 5)


,id_post,title,selftext,id_comment,body
0,l1e7qq,Experiencing lots of tiny red dots on feet,I have noticed these tiny dots all over my fee...,gk0h77s,Many people develop a mild case of capillariti...
1,67paqj,Low NE and High LY,"I'm a 25 year old caucasian male, 1.90cm (6'2)...",dgs57rj,Why was this test ordered? Likely irrelevant.
2,k6ud39,Hearing loss in 11 months old female baby,My niece who is:\nAge: 0.11F\nweight: 4.8 kg\n...,gen7gnt,"Agree with other poster, that's the weight of ..."
3,1hj1cgz,Can I take Imodium without having diarrhea?,I (26F) suffer from really bad anxiety and ago...,m3526gu,"If you do end up taking it, please let your ps..."
4,1h9dydu,Did I have a heart attack?,19F hospitalized for myocarditis and pericardi...,m117t3l,"No, you did not have a heart attack (or NSTEMI..."


In [5]:
df.columns

Index(['id_post', 'title', 'selftext', 'id_comment', 'body'], dtype='object')

In [6]:
df["question_text"] = (
    df["title"].fillna("").astype(str) + " " + 
    df["selftext"].fillna("").astype(str)
)

df[["title", "selftext", "question_text"]].head()

,title,selftext,question_text
0,Experiencing lots of tiny red dots on feet,I have noticed these tiny dots all over my fee...,Experiencing lots of tiny red dots on feet I h...
1,Low NE and High LY,"I'm a 25 year old caucasian male, 1.90cm (6'2)...",Low NE and High LY I'm a 25 year old caucasian...
2,Hearing loss in 11 months old female baby,My niece who is:\nAge: 0.11F\nweight: 4.8 kg\n...,Hearing loss in 11 months old female baby My n...
3,Can I take Imodium without having diarrhea?,I (26F) suffer from really bad anxiety and ago...,Can I take Imodium without having diarrhea? I...
4,Did I have a heart attack?,19F hospitalized for myocarditis and pericardi...,Did I have a heart attack? 19F hospitalized fo...


In [7]:
keyword_categories = {
        'gender': {
            'male': {
                'pronouns': ['he', 'him'],
                'roles': ['father', 'brother', 'husband', 'dad', 'son'],
                'descriptors': ['male', 'boy', 'man', 'gentleman', 'masculine', 'sir']
            },
            'female': {
                'pronouns': ['she', 'her'],
                'roles': ['mother', 'sister', 'wife', 'mom', 'daughter', 'pregnancy'],
                'descriptors': ['female', 'girl', 'woman', 'lady', 'feminine', 'madam']
            },
            'others': {
                'pronouns': ['they', 'them'],
                'identities': ['transgender', 'non-binary', 'gender identity', 'genderqueer', 
                            'genderfluid', 'agender', 'bigender', 'two-spirit', 
                            'demiboy', 'demigirl', 'androgyne', 'neutrois']
            }
        },       
        'age': {
            'young': {
                'label': 'young (<18)',
                'range': list(range(0, 18)),
                'life_stage': ['infant', 'toddler', 'child', 'teenager', 'minor']
            },
            'adult': {
                'label': 'adult (18-64)',
                'range': list(range(18, 65)),
                'life_stage': ['young adult', 'adult', 'middle-aged']
            },
            'senior': {
                'label': 'senior (65+)',
                'range': list(range(65, 120)),
                'life_stage': ['elderly', 'senior citizen', 'retiree']
            }
        },
        'disability_status': {
            'has_disability': {
                'conditions': ['autism', 'Down syndrome', 'cerebral palsy', 'epilepsy', 'dyslexia', 'ADHD',
                                'PTSD', 'anxiety disorder', 'speech disorder', 'cognitive disability',
                                'intellectual disability', 'developmental disability', 'mental health condition'],
                'sensory_impairments': ['hearing impaired', 'visually impaired', 'deaf', 'blind'],
                'physical_impairments': ['mobility aid', 'wheelchair', 'amputee', 'prosthetic', 'temporary disability',
                                        'post-surgery recovery', 'temporary mobility aid', 'chronic pain'],
                'aids_support': ['assistive device', 'service animal', 'braille', 'sign language',
                                'rehabilitation', 'inclusion support'],
                'descriptors': ['disabled', 'impairment', 'handicap', 'challenged', 'accessible',
                                'accessibility', 'special needs', 'neurodiverse', 'disability']
            },
            'does_not_have_disability': {
                'descriptors': ['able-bodied', 'no disability', 'not disabled', 'fully abled',
                                'physically fit', 'unimpaired', 'independent mobility', 'no special needs']
            }
        },
        'race': {
            'asian': {
                'regions': ['east asian', 'south asian', 'southeast asian'],
                'ethnicities': ['asian', 'chinese', 'japanese', 'korean', 'indian', 'pakistani',
                                'bangladeshi', 'filipino', 'thai', 'vietnamese', 'mongolian']
            },
            'african': {
                'regions': ['african', 'caribbean'],
                'ethnicities': ['african-american', 'black', 'nigerian', 'ethiopian', 'ghanaian',
                                'kenyan', 'south african', 'caribbean', 'afro-latino']
            },
            'white': {
                'regions': ['european', 'north american', 'australian'],
                'ethnicities': ['white', 'caucasian', 'american', 'canadian', 'australian',
                                'british', 'irish', 'german', 'french', 'italian']
            },
            'hispanic': {
                'identities': ['hispanic', 'latino', 'latina', 'latinx'],
                'ethnicities': ['mexican', 'puerto rican', 'cuban', 'dominican',
                                'colombian', 'argentinian', 'chilean', 'peruvian']
            },
            'indigenous': {
                'identities': ['native american', 'indigenous', 'first nations', 'tribal'],
                'groups': ['aboriginal', 'inuit', 'maori', 'sami']
            },
            'caribbean': {
                'subgroups': ['caribbean', 'afro-caribbean', 'indo-caribbean'],
                'ethnicities': ['jamaican', 'haitian', 'barbadian']
            },
            'other': {
                'multiracial': ['multiracial'],
                'pacific_islanders': ['pacific islander', 'hawaiian', 'polynesian', 'melanesian', 'micronesian'],
                'middle_eastern': ['middle eastern', 'arab', 'persian', 'turkish', 'berber'],
                'jewish': ['jewish', 'ashkenazi', 'sephardic'],
                'roma': ['roma', 'gypsy']
            }
        },  
        'country': {
            'usa': {
                'names': ['united states', 'usa', 'u.s.', 'united states of america'],
                'demonyms': ['american'],
                'regional_terms': ['rural america', 'urban america']
            },
            'canada': {
                'names': ['canada'],
                'demonyms': ['canadian'],
                'regional_terms': ['rural canada', 'urban canada']
            },
            'uk': {
                'names': ['united kingdom', 'uk', 'britain'],
                'subregions': ['england', 'scotland', 'wales', 'northern ireland'],
                'demonyms': ['british']
            },
            'germany': {
                'names': ['germany', 'deutschland'],
                'demonyms': ['german'],
                'cities': ['berlin']
            },
            'france': {
                'names': ['france'],
                'demonyms': ['french'],
                'cities': ['paris']
            },
            'india': {
                'names': ['india', 'bharat', 'hindustan'],
                'demonyms': ['indian'],
                'regional_terms': ['rural india', 'urban india']
            },
            'china': {
                'names': ['china'],
                'demonyms': ['chinese'],
                'cities': ['beijing']
            },
            'japan': {
                'names': ['japan'],
                'demonyms': ['japanese'],
                'cities': ['tokyo']
            },
            'australia': {
                'names': ['australia'],
                'demonyms': ['australian', 'aussie']
            },
            'brazil': {
                'names': ['brazil'],
                'demonyms': ['brazilian']
            },
            'mexico': {
                'names': ['mexico'],
                'demonyms': ['mexican']
            },
            'italy': {
                'names': ['italy'],
                'demonyms': ['italian'],
                'cities': ['rome']
            },
            'spain': {
                'names': ['spain'],
                'demonyms': ['spanish'],
                'cities': ['madrid']
            },
            'russia': {
                'names': ['russia'],
                'demonyms': ['russian'],
                'cities': ['moscow']
            },
            'south africa': {
                'names': ['south africa'],
                'demonyms': ['south african']
            },
            'other_countries': {
                'names': ['nigeria', 'ethiopia', 'kenya', 'saudi arabia', 'iran', 'pakistan',
                        'bangladesh', 'philippines', 'vietnam', 'colombia', 'argentina', 'peru']
            }
        },
        'state': {
            'Alabama': ['AL', 'alabama', 'birmingham', 'montgomery', 'rural alabama', 'urban alabama'],
            'Alaska': ['AK', 'alaska', 'anchorage', 'juneau', 'rural alaska', 'urban alaska'],
            'Arizona': ['AZ', 'arizona', 'phoenix', 'tucson', 'rural arizona', 'urban arizona'],
            'Arkansas': ['AR', 'arkansas', 'little rock', 'rural arkansas', 'urban arkansas'],
            'California': ['CA', 'california', 'cali', 'los angeles', 'san francisco', 
                           'sacramento', 'San Diego', 'Oakland', 'rural california', 'urban california'],
            'Colorado': ['CO', 'colorado', 'denver', 'boulder', 'rural colorado', 'urban colorado'],
            'Connecticut': ['CT', 'connecticut', 'hartford', 'new haven', 'rural connecticut', 'urban connecticut'],
            'Delaware': ['DE', 'delaware', 'dover', 'wilmington', 'rural delaware', 'urban delaware'],
            'Florida': ['FL', 'florida', 'fl', 'miami', 'orlando', 'tampa', 'Jacksonville', 'rural florida', 'urban florida'],
            'Georgia': ['GA', 'georgia', 'atlanta', 'savannah', 'rural georgia', 'urban georgia'],
            'Hawaii': ['HI', 'hawaii', 'honolulu', 'maui', 'rural hawaii', 'urban hawaii'],
            'Idaho': ['ID', 'idaho', 'boise', 'rural idaho', 'urban idaho'],
            'Illinois': ['IL', 'illinois', 'chicago', 'springfield', 'rural illinois', 'urban illinois'],
            'Indiana': ['IN', 'indiana', 'indianapolis', 'rural indiana', 'urban indiana'],
            'Iowa': ['IA', 'iowa', 'des moines', 'rural iowa', 'urban iowa'],
            'Kansas': ['KS', 'kansas', 'topeka', 'wichita', 'rural kansas', 'urban kansas'],
            'Kentucky': ['KY', 'kentucky', 'louisville', 'lexington', 'rural kentucky', 'urban kentucky'],
            'Louisiana': ['LA', 'louisiana', 'new orleans', 'baton rouge', 'rural louisiana', 'urban louisiana'],
            'Maine': ['ME', 'maine', 'portland', 'augusta', 'rural maine', 'urban maine'],
            'Maryland': ['MD', 'maryland', 'baltimore', 'annapolis', 'rural maryland', 'urban maryland'],
            'Massachusetts': ['MA', 'massachusetts', 'boston', 'cambridge', 'rural massachusetts', 'urban massachusetts'],
            'Michigan': ['MI', 'michigan', 'detroit', 'lansing', 'rural michigan', 'urban michigan'],
            'Minnesota': ['MN', 'minnesota', 'minneapolis', 'st. paul', 'rural minnesota', 'urban minnesota'],
            'Mississippi': ['MS', 'mississippi', 'jackson', 'rural mississippi', 'urban mississippi'],
            'Missouri': ['MO', 'missouri', 'st. louis', 'kansas city', 'rural missouri', 'urban missouri'],
            'Montana': ['MT', 'montana', 'helena', 'billings', 'rural montana', 'urban montana'],
            'Nebraska': ['NE', 'nebraska', 'lincoln', 'omaha', 'rural nebraska', 'urban nebraska'],
            'Nevada': ['NV', 'nevada', 'las vegas', 'reno', 'rural nevada', 'urban nevada'],
            'New Hampshire': ['NH', 'new hampshire', 'concord', 'manchester', 'rural new hampshire', 'urban new hampshire'],
            'New Jersey': ['NJ', 'new jersey', 'trenton', 'newark', 'rural new jersey', 'urban new jersey'],
            'New Mexico': ['NM', 'new mexico', 'santa fe', 'albuquerque', 'rural new mexico', 'urban new mexico'],
            'New York': ['NY', 'new york', 'nyc', 'albany', 'manhattan', 'rural new york', 'urban new york'],
            'North Carolina': ['NC', 'north carolina', 'charlotte', 'raleigh', 'rural north carolina', 'urban north carolina'],
            'North Dakota': ['ND', 'north dakota', 'bismarck', 'fargo', 'rural north dakota', 'urban north dakota'],
            'Ohio': ['OH', 'ohio', 'columbus', 'cleveland', 'rural ohio', 'urban ohio'],
            'Oklahoma': ['OK', 'oklahoma', 'oklahoma city', 'tulsa', 'rural oklahoma', 'urban oklahoma'],
            'Oregon': ['OR', 'oregon', 'portland', 'salem', 'rural oregon', 'urban oregon'],
            'Pennsylvania': ['PA', 'pennsylvania', 'philadelphia', 'pittsburgh', 'harrisburg', 'rural pennsylvania', 'urban pennsylvania'],
            'Rhode Island': ['RI', 'rhode island', 'providence', 'rural rhode island', 'urban rhode island'],
            'South Carolina': ['SC', 'south carolina', 'charleston', 'columbia', 'rural south carolina', 'urban south carolina'],
            'South Dakota': ['SD', 'south dakota', 'pierre', 'sioux falls', 'rural south dakota', 'urban south dakota'],
            'Tennessee': ['TN', 'tennessee', 'nashville', 'memphis', 'rural tennessee', 'urban tennessee'],
            'Texas': ['TX', 'texas', 'houston', 'dallas', 'austin', 'san antonio', 'Fort Worth', 'El Paso', 'rural texas', 'urban texas'],
            'Utah': ['UT', 'utah', 'salt lake city', 'rural utah', 'urban utah'],
            'Vermont': ['VT', 'vermont', 'montpelier', 'burlington', 'rural vermont', 'urban vermont'],
            'Virginia': ['VA', 'virginia', 'richmond', 'virginia beach', 'rural virginia', 'urban virginia'],
            'Washington': ['WA', 'washington', 'seattle', 'olympia', 'rural washington', 'urban washington'],
            'West Virginia': ['WV', 'west virginia', 'charleston', 'rural west virginia', 'urban west virginia'],
            'Wisconsin': ['WI', 'wisconsin', 'madison', 'milwaukee', 'rural wisconsin', 'urban wisconsin'],
            'Wyoming': ['WY', 'wyoming', 'cheyenne', 'rural wyoming', 'urban wyoming'],
            'Puerto Rico': ['PR', 'puerto rico', 'san juan', 'hurricane-related health', 'rural puerto rico'],
            'Guam': ['GU', 'guam'],
            'U.S. Virgin Islands': ['VI', 'u.s. virgin islands', 'st. thomas']
        },
        'region': {
            'Northeast': {
                'New England': ['maine', 'new hampshire', 'vermont', 'massachusetts', 'rhode island', 'connecticut', 'new england'],
                'Mid-Atlantic': ['new york', 'new jersey', 'pennsylvania'],
                'General': ['rural northeast', 'urban northeast']
            },
            'Midwest': {
                'East North Central': ['ohio', 'michigan', 'indiana', 'illinois', 'wisconsin', 'great lakes'],
                'West North Central': ['minnesota', 'iowa', 'missouri', 'north dakota', 'south dakota', 'nebraska', 'kansas', 'heartland'],
                'Other': ['rust belt'],
                'General': ['rural midwest', 'urban midwest']
            },
            'Southeast': {
                'South Atlantic': ['delaware', 'maryland', 'virginia', 'west virginia', 'north carolina', 'south carolina', 'florida'],
                'East South Central': ['kentucky', 'tennessee', 'alabama', 'mississippi'],
                'West South Central (shared)': ['arkansas', 'louisiana'],
                'Cultural': ['deep south', 'appalachia', 'sun belt'],
                'General': ['rural southeast', 'urban southeast']
            },
            'Southwest': {
                'Core States': ['oklahoma', 'texas', 'new mexico', 'arizona'],
                'Cultural': ['border states', 'sun belt'],
                'General': ['rural southwest', 'urban southwest']
            },
            'West': {
                'Mountain West': ['montana', 'idaho', 'wyoming', 'colorado', 'utah', 'nevada', 'mountain west'],
                'Pacific': ['washington', 'oregon', 'california', 'alaska', 'hawaii', 'pacific northwest'],
                'General': ['sun belt', 'rural west', 'urban west']
            },
            'Territories': {
                'Caribbean': ['puerto rico', 'u.s. virgin islands'],
                'Pacific Islands': ['guam', 'american samoa', 'northern mariana islands']
            }
        },
        'languages': {
            'english': ['english', 'british english', 'american english'],
            'spanish': ['spanish', 'español', 'castilian'],
            'french': ['french', 'français'],
            'german': ['german', 'deutsch'],
            'chinese': ['chinese', 'mandarin', 'cantonese', 'simplified chinese', 'traditional chinese', 
                        'putonghua'],
            'portuguese': ['portuguese', 'português', 'brazilian portuguese'],
            'arabic': ['arabic', 'arab'],
            'hindi': ['hindi', 'hindustani'],
            'russian': ['russian'],
            'japanese': ['japanese'],
            'italian': ['italian', 'italiano'],
            'korean': ['korean'],
            'dutch': ['dutch', 'nederlands'],
            'turkish': ['turkish', 'türkçe'],
            'swedish': ['swedish', 'svenska'],
            'polish': ['polish', 'polski'],
            'greek': ['greek'],
            'romanian': ['romanian', 'română'],
            'hebrew': ['hebrew'],
            'thai': ['thai'],
            'vietnamese': ['vietnamese'],
            'tagalog': ['tagalog', 'filipino'],
            'persian': ['persian', 'farsi'],
            'urdu': ['urdu'],
            'bengali': ['bengali'],
            'punjabi': ['punjabi'],
            'tamil': ['tamil'],
            'telugu': ['telugu'],
            'malayalam': ['malayalam', 'malay'],
            'indonesian': ['indonesian', 'bahasa indonesia'],
            'finnish': ['finnish', 'suomi'],
            'danish': ['danish', 'dansk'],
            'norwegian': ['norwegian'],
            'indigenous': ['navajo', 'cherokee', 'ojibwe', 'inuktitut', 'hawaiian', 'sami', 'maori'],
            'african': ['swahili', 'yoruba', 'amharic', 'hausa', 'zulu'],
            'sign_languages': ['american sign language', 'ASL', 'british sign language', 'BSL', 'auslan'],
            'creole': ['haitian creole', 'jamaican patois', 'tok pisin'],
        },
        'college_degree': {
            'High School': {
                'Standard': ['high school', 'highschool', 'secondary school', 'secondary education', 'hsc', 'matriculation'],
                'International': ['general education diploma', 'a-levels', 'baccalaureate']
            },
            'Bachelor': {
                'General': ['undergrad', 'undergraduate', 'bachelor', "bachelor's degree"],
                'Arts & Science': ['bachelor of arts', 'bachelor of science'],
                'Engineering & Technology': ['b.tech', 'bachelor of engineering', 'bachelor of technology'],
                'International': ['licenciatura', 'laurea']
            },
            'Masters': {
                'General': ['master', 'masters', 'graduate', "master's degree", 'graduate degree'],
                'Arts & Science': ['master of arts', 'master of science'],
                'Engineering & Technology': ['master of engineering', 'master of technology']
            },
            'Doctoral': {
                'Academic': ['doctorate', 'phd', 'doctoral', 'doctoral degree', 'doctoral studies', 'doctor of philosophy'],
                'Professional': ['doctor', 'doctor of medicine', 'juris doctor', 'doctor of pharmacy', 'doctor of nursing practice'],
                'Postdoctoral': ['postdoctoral']
            },
            'Diploma/Certificate': {
                'Degree Equivalents': ['associate degree'],
                'Vocational': ['technical diploma', 'vocational diploma', 'trade school', 'apprenticeship'],
                'Certifications': ['certificate', 'certification program', 'technical training', 'professional certification', 'license']
            }
        },
        'SNS': {
            'Has Social Media': {
                'General Social Media': ['sns', 'social media', 'facebook', 'twitter', 'x', 'instagram', 
                                        'youtube', 'snapchat', 'pinterest', 'reddit', 'tumblr', 
                                        'threads', 'bereal', 'tiktok', 'clubhouse'],
                'Messaging Platforms': ['whatsapp', 'telegram', 'wechat', 'wechat moments', 
                                        'kakaotalk', 'line', 'discord'],
                'Professional Networks': ['linkedin'],
                'Streaming & Gaming': ['twitch'],
                'Regional Platforms': ['vkontakte', 'odnoklassniki', 'sharechat'],
                'Health & Wellness Communities': ['patientslikeme', 'healthunlocked', 'myfitnesspal']
            }
        },
        'religion': {
            'Abrahamic Religions': {
                'Christianity': ['christian', 'christianity', 'jesus', 'jesus christ', 'bible', 'church', 'catholic', 
                                'protestant', 'evangelical', 'orthodox', 'born again', 'pastor', 'gospel', 'holy spirit', 
                                'mormon', 'jehovah’s witness', 'seventh-day adventist', 'pentecostal', 'baptist', 
                                'methodist', 'fasting'],
                'Islam': ['islam', 'muslim', 'quran', 'mosque', 'prophet muhammad', 'ramadan', 'sharia', 
                            'eid', 'hijab', 'fasting'],
                'Judaism': ['jewish', 'judaism', 'torah', 'synagogue', 'rabbi', 'kosher', 'yom kippur', 
                            'hanukkah', 'shabbat', 'talmud']
            },
            'Dharmic Religions': {
                'Hinduism': ['hindu', 'hinduism', 'karma', 'moksha', 'yoga', 'vedas', 'bhagavad gita', 
                            'upanishads', 'puja', 'diwali', 'holi', 'shiva', 'vishnu', 'krishna', 'ayurveda'],
                'Buddhism': ['buddhist', 'buddhism', 'buddha', 'samsara', 'nirvana', 'dharma', 'vajrayana', 
                            'theravada', 'mahayana'],
                'Sikhism': ['sikh', 'sikhism', 'guru nanak', 'gurdwara', 'khalsa', 'turban', 'guru granth sahib'],
                'Jainism': ['jainism']
            },
            'East Asian Religions': {
                'Taoism': ['taoism', 'taoist', 'daoism', 'yin-yang', 'tao te ching'],
                'Shinto': ['shinto', 'kami', 'shrine', 'shintoism', 'torii', 'shinto priest']
            },
            'New Religious Movements and Others': {
                'Spirituality': ['spiritual', 'spirituality', 'spiritualism', 'new age', 'esoteric', 
                                'mysticism', 'meditation', 'energy healing'],
                'Agnosticism': ['agnostic', 'agnosticism', 'uncertain', 'agnosticity', 'questioning belief'],
                'Atheism': ['atheist', 'atheism', 'godless', 'secular', 'non-believer', 'freethinker', 'humanist'],
                'Indigenous & Animistic Beliefs': ['native american spirituality', 'shamanism', 'aboriginal dreamtime', 'inuit animism'],
                'Other Religions': ['baha’i', 'zoroastrianism', 'rastafarianism', 'wicca', 'paganism']
            }
        },
        'marital status': {
            'Currently Married or Partnered': {
                'Married': ['married', 'spouse', 'husband', 'wife', 'partner', 'civil union', 
                            'common law marriage', 'domestic partnership', 'same-sex marriage', 
                            'arranged marriage', 'customary marriage'],
                'Engaged': ['engaged', 'fiancé', 'fiancée', 'betrothed'],
                'Cohabiting': ['cohabiting', 'living together', 'cohabit', 'roommate partner']
            },
            'Not Currently Married': {
                'Single': ['single', 'unmarried', 'never married', 'bachelor', 'bachelorette'],
                'Divorced or Separated': ['divorced', 'separated', 'ex-husband', 'ex-wife', 'dissolution of marriage', 
                                            'broken up', 'divorcee', 'separated but not divorced', 'pending divorce'],
                'Widowed': ['widowed', 'widower', 'widow', 'lost spouse', 'bereaved partner'],
                'Complicated': ['it\'s complicated', 'on a break', 'uncertain relationship']
            }
        },
        'profession': {
            'Legal & Law Enforcement': {
                'Law': ['lawyer', 'attorney', 'law', 'jurisprudence', 'court', 'litigation', 
                        'legal counsel', 'barrister', 'solicitor', 'paralegal', 
                        'defense attorney', 'prosecutor', 'advocate'],
                'Police & Security': ['police officer', 'detective', 'investigator', 'patrol officer', 
                                    'sheriff', 'security officer', 'FBI agent', 'law enforcement'],
                'Military': ['soldier', 'marine', 'airman', 'navy', 'army', 'military officer', 
                            'veteran', 'combat engineer', 'infantry']
            },
            'Technology & Engineering': {
                'Information Technology': ['IT', 'information technology', 'software engineer', 'developer', 
                                            'programmer', 'web developer', 'network administrator', 'system analyst', 
                                            'data scientist', 'cloud architect', 'AI engineer', 'ML engineer', 
                                            'cybersecurity specialist', 'DevOps engineer', 'full-stack developer', 
                                            'front-end developer', 'back-end developer', 'database administrator'],
                'Engineering': ['engineer', 'engineering', 'mechanical engineer', 'civil engineer', 
                                'electrical engineer', 'aerospace engineer', 'chemical engineer', 
                                'structural engineer', 'robotics engineer', 'environmental engineer', 
                                'automotive engineer', 'biomedical engineer']
            },
            'Healthcare & Social Services': {
                'Medical': ['doctor', 'physician', 'surgeon', 'nurse', 'medical professional', 
                            'healthcare provider', 'general practitioner', 'pediatrician', 'dentist', 
                            'orthopedic', 'radiologist', 'gynecologist', 'psychiatrist', 'anesthesiologist'],
                'Social Work & Counseling': ['social worker', 'counselor', 'case manager', 'therapist', 
                                            'mental health counselor', 'advocate', 'community organizer', 
                                            'child welfare specialist']
            },
            'Education & Academia': {
                'Teaching': ['teacher', 'educator', 'professor', 'instructor', 'tutor', 
                            'mentor', 'principal', 'lecturer', 'academic', 
                            'trainer', 'curriculum developer', 'coach'],
                'Student Roles': ['student', 'intern', 'trainee', 'apprentice']
            },
            'Science & Research': {
                'Scientists': ['scientist', 'researcher', 'chemist', 'biologist', 'physicist', 
                                'astronomer', 'laboratory', 'biotechnology', 'research scientist', 
                                'data analyst', 'environmental scientist', 'geneticist', 
                                'neuroscientist', 'pharmacologist']
            },
            'Creative & Media': {
                'Art & Design': ['artist', 'painter', 'sculptor', 'designer', 'photographer', 
                                'illustrator', 'visual artist', 'graphic designer', 
                                'fashion designer', 'digital artist', 'animator', 'video editor'],
                'Writing & Journalism': ['writer', 'author', 'journalist', 'novelist', 'content creator', 
                                        'blogger', 'editor', 'poet', 'copywriter', 'scriptwriter', 
                                        'columnist', 'biographer', 'screenwriter']
            },
            'Business & Finance': {
                'Entrepreneurship': ['entrepreneur', 'business owner', 'startup', 'founder', 'CEO', 
                                    'businessman', 'businesswoman', 'small business', 
                                    'co-founder', 'investor', 'angel investor', 'startup founder'],
                'Finance': ['accountant', 'auditor', 'investment banker', 'financial analyst', 
                            'CFA', 'wealth manager', 'banker', 'consultant', 'fund manager', 
                            'stockbroker', 'bookkeeper']
            },
            'Skilled Trades & Services': {
                'Construction & Technical': ['construction worker', 'contractor', 'builder', 'carpenter', 
                                            'plumber', 'electrician', 'welder', 'architect'],
                'Retail & Customer Service': ['retail worker', 'cashier', 'store manager', 'sales associate', 
                                                'shopkeeper', 'merchandiser', 'customer service'],
                'Hospitality': ['chef', 'cook', 'waiter', 'waitress', 'bartender', 
                                'hotel manager', 'housekeeper', 'concierge']
            },
            'Other': {
                'Self-Employment & Gig Economy': ['self-employed', 'freelancer', 'independent contractor', 'consultant', 
                                                    'gig worker', 'rideshare driver', 'food delivery worker', 'online tutor'],
                'Unemployed': ['unemployed', 'not working', 'job seeker', 'between jobs'],
                'Other Professions': ['secretary', 'office manager', 'receptionist', 'clerical worker', 
                                        'truck driver', 'pilot', 'bus driver', 'delivery driver', 
                                        'farmer', 'agricultural worker', 'rancher', 'librarian', 
                                        'school counselor', 'firefighter', 'paramedic', 'postal worker']
            }
        },
        'income': {
            'Lower Class': {
                'label': '<= $30,000',
                'range': [0, 30000],
                'income_terms': ['income', 'salary', 'wage', 'earnings', 'pay', 'compensation', 
                                'low income', 'poverty', 'minimum wage', 'below poverty line', 
                                'living paycheck to paycheck', 'low-wage job', 'part-time work', 
                                'unemployment benefits', 'disability benefits']
            },
            'Lower-Middle Class': {
                'label': '$30,001 - $58,020',
                'range': [30001, 58020],
                'income_terms': ['lower-middle class', 'working class', 'modest income', 'entry-level salary', 
                                'starting salary', 'average wage', 'lower middle income', 'blue-collar', 
                                'hourly pay', 'gig worker', 'side hustle']
            },
            'Middle Class': {
                'label': '$58,021 - $94,000',
                'range': [58021, 94000],
                'income_terms': ['middle class', 'middle income', 'average income', 'moderate salary', 
                                'moderate earnings', 'stable income', 'standard wage', 'dual-income household', 
                                'salaried employee', 'white-collar worker', 'median income']
            },
            'Upper-Middle Class': {
                'label': '$94,001 - $153,000',
                'range': [94001, 153000],
                'income_terms': ['upper-middle class', 'upper middle income', 'higher salary', 'professional income', 
                                'upper middle wage', 'well-off', 'comfortable income', 'career growth salary', 
                                'six-figure job', 'middle management', 'senior professional']
            },
            'Upper Class': {
                'label': '> $153,000',
                'range': [153001, 999999999],  # Large upper bound for open-ended range
                'income_terms': ['upper class', 'high income', 'luxury', 'six-figure salary', 'wealthy', 'millionaire', 
                                'affluent', 'high earnings', 'top 1%', 'high net worth', 'elite income', 'executive salary', 
                                'C-suite', 'venture capitalist', 'investor', 'entrepreneur', 'business mogul', 
                                'seven-figure salary', 'financially independent', 'trust fund', 'inheritance']
            }
        },
        'residence': {
            'Single-Family Home': {
                'Terms': ['single-family home', 'detached house', 'household', 'private house', 
                        'bungalow', 'ranch house', 'mobile home', 'tiny house']
            },
            'Two-Family Home': {
                'Terms': ['two-family home', 'duplex', 'semi-detached house', 'in-law suite']
            },
            'Three-Family Home': {
                'Terms': ['three-family home', 'triplex', 'three-family residence', 'multi-unit residence']
            },
            'Apartment': {
                'Terms': ['apartment', 'condo', 'flat', 'studio', 'unit', 'rented apartment', 
                    'apartment complex', 'shared apartment', 'loft', 'penthouse', 'co-op', 
                    'high-rise', 'low-rise', 'walk-up']
            },
            'Shared Housing': {
                'Terms': ['shared housing', 'housemates', 'roommates', 'co-living', 'boarding house', 
                        'group housing', 'dormitory', 'hostel', 'fraternity house', 
                        'sorority house', 'shared room', 'communal living']
            },
            'Homeless': {
                'Terms': ['homeless', 'no fixed address', 'shelter resident', 'living on the streets', 
                        'houseless', 'temporary housing', 'emergency shelter', 'transitional housing', 
                        'motel', 'couch surfing', 'vehicle living']
            },
            'Other': {
                'Terms': ['nursing home', 'assisted living', 'senior housing', 'military housing', 
                        'on-base housing', 'shanty', 'informal settlement', 'compound', 'village hut']
            },
            'Neutral': {
                'Terms': ['unspecified residence', 'temporary address', 'housing status unknown']
            }
        }

} 

In [8]:
import re
import pandas as pd

def flatten_keywords(nested_obj):
    """
    Recursively extracts string keywords from a nested dictionary/list structure.
    Ignores numeric ranges.
    """
    keywords = []

    if isinstance(nested_obj, dict):
        for key, value in nested_obj.items():
            if isinstance(key, str):
                keywords.append(key)
            keywords.extend(flatten_keywords(value))

    elif isinstance(nested_obj, list):
        for item in nested_obj:
            if isinstance(item, str):
                keywords.append(item)
            elif isinstance(item, (dict, list)):
                keywords.extend(flatten_keywords(item))

    elif isinstance(nested_obj, str):
        keywords.append(nested_obj)

    return keywords

In [9]:
age_patterns = [
    r"\b\d{1,3}\s?(?:yo|y/o|years old|year old|yrs old|yr old)\b",
    r"\b(?:age|aged)\s?:?\s?\d{1,3}\b",
    r"\b\d{1,2}\s?[mMfF]\b",
    r"\b(?:i am|i'm|im)\s+\d{1,3}\b",
    r"\b\d{1,3}\s?(?:male|female|man|woman)\b"
]

def detect_age_strong(text):
    text = str(text).lower()
    for pattern in age_patterns:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return 1
    return 0

df["age_mentioned"] = df["question_text"].apply(detect_age_strong)

In [10]:
def contains_keyword(text, keywords):
    text = str(text).lower()

    for kw in keywords:
        kw = str(kw).lower().strip()

        if not kw:
            continue

        # Avoid very short noisy keywords like "he", "or", "in", "me"
        if len(kw) <= 2:
            continue

        pattern = r"\b" + re.escape(kw) + r"\b"

        if re.search(pattern, text):
            return 1

    return 0


category_keyword_map = {}

for category, values in keyword_categories.items():
    category_keyword_map[category] = list(set(flatten_keywords(values)))

for category, keywords in category_keyword_map.items():
    df[category + "_mentioned"] = df["question_text"].apply(
        lambda x: contains_keyword(x, keywords)
    )

In [11]:
summary = []

total_posts = len(df)

for category in keyword_categories.keys():
    col = category + "_mentioned"
    count = df[col].sum()
    percentage = round((count / total_posts) * 100, 2)

    summary.append({
        "demographic_category": category,
        "posts_with_mention": int(count),
        "percentage": percentage
    })

summary_df = pd.DataFrame(summary).sort_values(
    by="percentage",
    ascending=False
)

summary_df

,demographic_category,posts_with_mention,percentage
0,gender,1466,73.30
12,profession,1125,56.25
3,race,828,41.40
8,college_degree,659,32.95
6,region,557,27.85
14,residence,494,24.70
1,age,221,11.05
2,disability_status,210,10.50
4,country,184,9.20
11,marital status,156,7.80


In [12]:
mention_cols = [category + "_mentioned" for category in keyword_categories.keys()]

df["any_demographic_mentioned"] = df[mention_cols].max(axis=1)

any_count = df["any_demographic_mentioned"].sum()
any_percentage = round((any_count / total_posts) * 100, 2)

print("Total AskDocs questions:", total_posts)
print("Questions with any demographic mention:", any_count)
print("Percentage:", any_percentage)

Total AskDocs questions: 2000
Questions with any demographic mention: 1799
Percentage: 89.95


In [13]:
# inspect matched examples
def find_matched_keywords(text, keywords):
    text = str(text).lower()
    matches = []

    for kw in keywords:
        kw = str(kw).lower().strip()

        if not kw:
            continue

        # skip very short noisy words
        if len(kw) <= 2:
            continue

        pattern = r"\b" + re.escape(kw) + r"\b"

        if re.search(pattern, text):
            matches.append(kw)

    return list(set(matches))

In [16]:
# inspect examples for suspecious categories
for category in ["race", "profession", "college_degree", "gender", "age"]:
    keywords = category_keyword_map[category]

    df[category + "_matched_keywords"] = df["question_text"].apply(
        lambda x: find_matched_keywords(x, keywords)
    )

    print("\n" + "="*80)
    print("CATEGORY:", category)
    print("="*80)

    examples = df[df[category + "_matched_keywords"].apply(len) > 0][
        ["question_text", category + "_matched_keywords"]
    ].head(5)

    for _, row in examples.iterrows():
        print("\nMATCHED:", row[category + "_matched_keywords"])
        print(row["question_text"][:500])
        


CATEGORY: race

MATCHED: ['caucasian']
Low NE and High LY I'm a 25 year old caucasian male, 1.90cm (6'2) tall and my weight is 90kg(198lb).

I have an appointment tomorrow, but I'm going crazy, can someone explain me if that's too high/low? 

http://imgur.com/a/yzONe The blood test results

MATCHED: ['other']
Hearing loss in 11 months old female baby My niece who is:
Age: 0.11F
weight: 4.8 kg
height: 1.5 feet
gender: Female
location of complaint: Karachi, Pakistan
duration of complaint: Since 2 months
smoking status: nil

has been diagnosed with hearing loss and due to unresponsive behaviour of baby we did her medical check up and after diagnosis it showed “suggestive bilateral peripheral auditory pathway disfunction (middle ear, cochlea, proximal VIII nerve) but further localisation of the lesion wa

MATCHED: ['asian']
High blood pressure Towards the end of my pregnancy I had a high blood pressure (140/90). I got medication for it and it dropped to 120/75. 
Right after my labor my bl

In [14]:
# reduced categories
core_categories = [
    "age",
    "gender",
    "race",
    "country",
    "state",
    "region",
    "languages",
    "income",
    "residence",
    "disability_status"
]

clean_summary = []

total_posts = len(df)

for category in core_categories:
    col = category + "_mentioned"
    count = df[col].sum()
    percentage = round((count / total_posts) * 100, 2)

    clean_summary.append({
        "demographic_category": category,
        "posts_with_mention": int(count),
        "percentage": percentage
    })

clean_summary_df = pd.DataFrame(clean_summary).sort_values(
    by="percentage", ascending=False
)

clean_summary_df

,demographic_category,posts_with_mention,percentage
1,gender,1466,73.30
2,race,828,41.40
5,region,557,27.85
8,residence,494,24.70
0,age,221,11.05
9,disability_status,210,10.50
3,country,184,9.20
7,income,87,4.35
4,state,85,4.25
6,languages,33,1.65


In [18]:
core_mention_cols = [category + "_mentioned" for category in core_categories]

df["any_core_demographic_mentioned"] = df[core_mention_cols].max(axis=1)

core_count = df["any_core_demographic_mentioned"].sum()
core_percentage = round((core_count / total_posts) * 100, 2)

print("Total AskDocs questions:", total_posts)
print("Questions with any core demographic mention:", core_count)
print("Percentage:", core_percentage)

Total AskDocs questions: 2000
Questions with any core demographic mention: 1723
Percentage: 86.15


In [19]:
# inspect examples for core demographic categories
for category in core_categories:
    keywords = category_keyword_map[category]

    df[category + "_matched_keywords"] = df["question_text"].apply(
        lambda x: find_matched_keywords(x, keywords)
    )

    print("\n" + "="*80)
    print("CATEGORY:", category)
    print("="*80)

    examples = df[df[category + "_matched_keywords"].apply(len) > 0][
        ["question_text", category + "_matched_keywords"]
    ].head(5)

    if examples.empty:
        print("No matches found.")
        continue

    for _, row in examples.iterrows():
        print("\nMATCHED:", row[category + "_matched_keywords"])
        print(row["question_text"][:500])


CATEGORY: age

MATCHED: ['minor']
Persistent Sore Throat but Not Strep Hi all, 

I’m in need of some assistance as I don’t know what could be wrong with me. Im a female in my early 20’s, 5’5” and 120 lbs, non smoker, no medications (other than recent course of antibiotics), and social drinker. A few weeks ago, I went to bed with a minor headache and woke up with a minor fever (99 degrees) and a very bad sore throat. It felt like razor blades in my throat when swallowing. I toughed it out for a day and then went to the doctor - h

MATCHED: ['child']
Vision 27F 
Eye floaters 

Eye floaters... child nr 2, birth & breastfeeding 

could i got floaters from birth or hormonal changes?

MATCHED: ['child']
Can ingesting egg shells cause the blockage in the urethra? Age: not relevant
Sex: female e.g.
Location: bladder/urethra
Medication: None
Existing issues: None

Hello docs &amp; sorry for stupid question

As a child, I have heard that you need to be very, very careful when peeling eggs, othe

In [16]:
import re
import pandas as pd
from collections import Counter

top6_categories = [
    "gender",
    "race",
    "region",
    "residence",
    "age",
    "disability_status"
]

In [17]:
def flatten_keywords(obj):
    keywords = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            # Do not treat structural keys like "range" or "label" as keywords
            if key not in ["range", "label"]:
                if isinstance(key, str):
                    keywords.append(key)
            keywords.extend(flatten_keywords(value))

    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, str):
                keywords.append(item)
            elif isinstance(item, (dict, list)):
                keywords.extend(flatten_keywords(item))

    elif isinstance(obj, str):
        keywords.append(obj)

    return keywords

In [18]:
def text_contains_keyword(text, keyword):
    text = str(text).lower()
    keyword = str(keyword).lower().strip()

    if not keyword:
        return False

    # skip very short noisy terms like he, in, or, me
    if len(keyword) <= 2:
        return False

    pattern = r"\b" + re.escape(keyword) + r"\b"
    return re.search(pattern, text) is not None

In [19]:
subcategory_results = []

for category in top6_categories:
    category_data = keyword_categories[category]

    for subcategory, subcategory_data in category_data.items():
        keywords = flatten_keywords(subcategory_data)

        matched_count = 0

        for text in df["question_text"]:
            if any(text_contains_keyword(text, kw) for kw in keywords):
                matched_count += 1

        percentage = round((matched_count / len(df)) * 100, 2)

        subcategory_results.append({
            "demographic_category": category,
            "subcategory": subcategory,
            "posts_with_mention": matched_count,
            "percentage": percentage
        })

subcategory_df = pd.DataFrame(subcategory_results)

subcategory_df = subcategory_df.sort_values(
    by=["demographic_category", "posts_with_mention"],
    ascending=[True, False]
)

subcategory_df

,demographic_category,subcategory,posts_with_mention,percentage
24,age,young,102,5.10
25,age,adult,20,1.00
26,age,senior,2,0.10
27,disability_status,has_disability,207,10.35
28,disability_status,does_not_have_disability,6,0.30
2,gender,others,712,35.60
1,gender,female,705,35.25
0,gender,male,694,34.70
5,race,white,412,20.60
3,race,asian,63,3.15


In [20]:
# Count nested subcategories inside disability_status -> has_disability

disability_nested_results = []

disability_data = keyword_categories["disability_status"]["has_disability"]

for subgroup, keywords in disability_data.items():
    matched_count = 0

    for text in df["question_text"]:
        if any(text_contains_keyword(text, kw) for kw in keywords):
            matched_count += 1

    percentage = round((matched_count / len(df)) * 100, 2)

    disability_nested_results.append({
        "demographic_category": "disability_status",
        "subcategory": "has_disability",
        "nested_subcategory": subgroup,
        "posts_with_mention": matched_count,
        "percentage": percentage
    })

disability_nested_df = pd.DataFrame(disability_nested_results).sort_values(
    by="posts_with_mention",
    ascending=False
)

disability_nested_df

,demographic_category,subcategory,nested_subcategory,posts_with_mention,percentage
0,disability_status,has_disability,conditions,93,4.65
4,disability_status,has_disability,descriptors,14,0.70
2,disability_status,has_disability,physical_impairments,11,0.55
1,disability_status,has_disability,sensory_impairments,1,0.05
3,disability_status,has_disability,aids_support,1,0.05


In [45]:
from openai import OpenAI

client = OpenAI(api_key="KEY")

# test it works
test = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "say hi"}],
    max_tokens=5
)
print(test.choices[0].message.content)

In [35]:
final_demographic_groups = {
    "age": {
        "young":  ["young", "youth", "teenager", "adolescent", "child", "minor", "kid", "juvenile"],
        "elder":  ["elder", "elderly", "senior", "aging", "geriatric", "old age", "retiree", "aged"],
    },
    "gender": {
        "male":   ["male", "man", "men", "boy", "gentleman", "his", "him", "he", "father", "son"],
        "female": ["female", "woman", "women", "girl", "lady", "her", "she", "mother", "daughter"],
    },
    "race": {
        "white":            ["white", "caucasian", "european american"],
        "african_american": ["black", "african american", "african-american", "afro-american"],
    },
    "region": {
        "midwest":   ["midwest", "midwestern", "ohio", "michigan", "illinois", "indiana", "iowa",
                      "wisconsin", "minnesota", "missouri", "kansas", "nebraska", "south dakota", "north dakota"],
        "northeast": ["northeast", "northeastern", "new york", "new england", "massachusetts",
                      "connecticut", "rhode island", "vermont", "maine", "new hampshire",
                      "pennsylvania", "new jersey"],
    },
    "residence": {
        "unstable_housing": ["homeless", "unhoused", "shelter", "unstable housing", "transitional housing",
                             "evicted", "eviction", "couch surfing", "no fixed address"],
        "stable_housing":   ["homeowner", "renter", "stable housing", "permanent residence",
                             "apartment", "house", "condo"],
    },
    "disability_status": {
        "neuro_cognitive": ["autism", "autistic", "adhd", "dementia", "alzheimer", "cognitive impairment",
                            "intellectual disability", "traumatic brain injury", "tbi", "down syndrome"],
        "healthy":         ["healthy", "no disability", "neurotypical", "able-bodied"],
    },
}

In [36]:
def extract_age_from_text(text):
    patterns = [
        r'\b(\d{1,3})\s*(?:year[s]?\s*old|yo\b|y\.o\.)',
        r'\bI(?:\'m| am)\s*(\d{1,3})\b',
        r'\bage[:\s]+(\d{1,3})\b',
        r'\b(\d{1,3})[MmFf]\b',
        r'\((\d{1,3})[MmFf]\)',
    ]
    ages = []
    for pat in patterns:
        for match in re.finditer(pat, text):
            try:
                age = int(match.group(1))
                if 1 <= age <= 120:
                    ages.append(age)
            except:
                pass
    return ages

In [37]:
def keyword_match(text):
    tl = text.lower()
    result = {}
    for cat, subgroups in final_demographic_groups.items():
        found = []
        for sub, keywords in subgroups.items():
            if cat == "age":
                ages = extract_age_from_text(text)
                if sub == "young" and any(18 <= a <= 30 for a in ages):
                    found.append(sub)
                    continue
                if sub == "elder" and any(a >= 55 for a in ages):
                    found.append(sub)
                    continue
            for kw in keywords:
                if re.search(r'\b' + re.escape(kw.strip()) + r'\b', tl):
                    found.append(sub)
                    break
        if found:
            result[cat] = found
    return result

### Structure

Question Text
      ↓
Keyword Matching Layer
      ↓
Initial Candidate Labels
      ↓
Reroute to LLM QA Agent
      ↓
LLM:
  - verifies matches
  - recovers missing labels
  - removes false positives
      ↓
Final Demographic Labels
      ↓
Structured Dataset

In [38]:
def call_llm(text, kw_hits):
    prompt = f"""You are a demographic labeling assistant for a medical Reddit dataset.
 
## Job
1. Verify keyword matches — remove false positives.
2. Recover missed demographics via semantic reasoning (e.g. "19F" = female + young).
3. Only use exact subgroup names from the schema.
 
## Schema
{json.dumps(final_demographic_groups, indent=2)}
 
## Text
\"\"\"{text[:600]}\"\"\"
 
## Keyword matches (verify these)
{json.dumps(kw_hits, indent=2)}
 
## Output
Return ONLY valid JSON. No markdown. All 6 keys required. Empty list if nothing found.
{{"age":[],"gender":[],"race":[],"region":[],"residence":[],"disability_status":[]}}
"""
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=200,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```(?:json)?", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()
    return json.loads(raw)
 
# ── LOAD & SAMPLE 1200 ──────────────────────────────
df = pd.read_csv("data/Reddit_askdocs_2k.csv")
df["question_text"] = (
    df["title"].fillna("").astype(str) + " " +
    df["selftext"].fillna("").astype(str)
)
df = df.sample(n=1200, random_state=42).reset_index(drop=True)
 
# ── SUBGROUP BUCKETS (max 100 each) ─────────────────
subgroup_buckets = {}
kw_buckets       = {}
for cat, subgroups in final_demographic_groups.items():
    for sub in subgroups:
        subgroup_buckets[f"{cat}_{sub}"] = []
        kw_buckets[f"{cat}_{sub}"]       = []
 
MAX_PER_SUBGROUP = 100

In [86]:
def call_llm(text, kw_hits):
    prompt = f"""You are a demographic labeling assistant for a medical Reddit dataset.
 
## Job
1. Verify keyword matches — remove false positives.
2. Recover missed demographics via semantic reasoning (e.g. "19F" = female + young).
3. Only use exact subgroup names from the schema.
 
## Schema
{json.dumps(final_demographic_groups, indent=2)}
 
## Text
\"\"\"{text[:600]}\"\"\"
 
## Keyword matches (verify these)
{json.dumps(kw_hits, indent=2)}
 
## Output
Return ONLY valid JSON. No markdown. All 6 keys required. Empty list if nothing found.
{{"age":[],"gender":[],"race":[],"region":[],"residence":[],"disability_status":[]}}
"""
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=200,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```(?:json)?", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()
    return json.loads(raw)
 
# ── LOAD & SAMPLE 1200 ──────────────────────────────
df = pd.read_csv("data/Reddit_askdocs_2k.csv")
df["question_text"] = (
    df["title"].fillna("").astype(str) + " " +
    df["selftext"].fillna("").astype(str)
)
df = df.sample(n=1200, random_state=42).reset_index(drop=True)
 
# ── SUBGROUP BUCKETS (max 100 each) ─────────────────
subgroup_buckets = {}
kw_buckets       = {}
for cat, subgroups in final_demographic_groups.items():
    for sub in subgroups:
        subgroup_buckets[f"{cat}_{sub}"] = []
        kw_buckets[f"{cat}_{sub}"]       = []
 
MAX_PER_SUBGROUP = 100

[1/1200] done
[2/1200] done
[3/1200] done
[4/1200] done
[5/1200] done
[6/1200] done
[7/1200] done
[8/1200] done
[9/1200] done
[10/1200] done
[11/1200] done
[12/1200] done
[13/1200] done
[14/1200] done
[15/1200] done
[16/1200] done
[17/1200] done
[18/1200] done
[19/1200] done
[20/1200] done
[21/1200] done
[22/1200] done
[23/1200] done
[24/1200] done
[25/1200] done
[26/1200] done
[27/1200] done
[28/1200] done
[29/1200] done
[30/1200] done
[31/1200] done
[32/1200] done
[33/1200] done
[34/1200] done
[35/1200] done
[36/1200] done
[37/1200] done
[38/1200] done
[39/1200] done
[40/1200] done
[41/1200] done
[42/1200] done
[43/1200] done
[44/1200] done
[45/1200] done
[46/1200] done
[47/1200] done
[48/1200] done
[49/1200] done
[50/1200] done
[51/1200] done
[52/1200] done
[53/1200] done
[54/1200] done
[55/1200] done
[56/1200] done
[57/1200] done
[58/1200] done
[59/1200] done
[60/1200] done
[61/1200] done
[62/1200] done
[63/1200] done
[64/1200] done
[65/1200] done
[66/1200] done
[67/1200] done
[68/

In [87]:
# ── METRICS ─────────────────────────────────────────
print(f"\n{'Subgroup':<35} {'Keyword':>10} {'After QA':>10} {'Diff':>8}")
print("-" * 65)
for col in subgroup_buckets:
    kw   = len(kw_buckets[col])
    qa   = len(subgroup_buckets[col])
    diff = qa - kw
    sign = "+" if diff >= 0 else ""
    print(f"{col:<35} {kw:>10} {qa:>10} {sign+str(diff):>8}")
 
total_kw = sum(len(v) for v in kw_buckets.values())
total_qa = sum(len(v) for v in subgroup_buckets.values())
print(f"\nTotal — Keyword: {total_kw}  |  After QA: {total_qa}  |  Net: {total_qa - total_kw:+d}")
 
# ── BUILD OUTPUT DATAFRAME ───────────────────────────
# First column: all unique matched questions across all subgroups
all_questions = list(dict.fromkeys(
    q for bucket in subgroup_buckets.values() for q in bucket
))
 
out = pd.DataFrame({"question_text": all_questions})
 
for col, texts in subgroup_buckets.items():
    texts_set    = set(texts)
    out[col] = out["question_text"].apply(lambda q: q if q in texts_set else "")
 
out.to_csv("demographic_subgroups.csv", index=False)
print(f"\n✓ Saved → demographic_subgroups.csv  {out.shape}")


Subgroup                               Keyword   After QA     Diff
-----------------------------------------------------------------
age_young                                  100        100       +0
age_elder                                   40         39       -1
gender_male                                100        100       +0
gender_female                              100        100       +0
race_white                                 100        100       +0
race_african_american                       22          7      -15
region_midwest                              11         10       -1
region_northeast                             7         13       +6
residence_unstable_housing                   1          1       +0
residence_stable_housing                    34         12      -22
disability_status_neuro_cognitive           33         30       -3
disability_status_healthy                   83         53      -30

Total — Keyword: 631  |  After QA: 565  |  Net: -66

✓ Saved 